# Serving

This notebook configures a deployment for the Breast Cancer schema without writing any repository checkpoints. It demonstrates serving mechanics, request validation, and target handling; it does not produce a useful trained classifier.

Serving starts with the same model schema used for training. The additional dependency is Pydantic, which defines the request object accepted by the deployment wrapper.


In [1]:
import os

import polars as pl
import pydantic
import torch
from loguru import logger

import json2vec as j2v
from json2vec.inference.deployment import Deployment

logger.remove()

The serving payload uses the same nested `measurements` shape as the nested supervised training tutorial, but omits the answer field. Keeping train and serve shapes aligned is the main reason schemas, queries, and preprocessors live together.


In [2]:
records = pl.read_ndjson("docs/data/breast-cancer.jsonl").head(32)


class Request(pydantic.BaseModel):
    measurements: list[dict]

The model includes a `diagnosis` target so it can be trained or inspected like the nested supervised training example. The request model validates the raw API payload before JSON2Vec encodes it.

In [3]:
model = j2v.Model.from_schema(
    j2v.Array(
        j2v.Category("name", max_vocab_size=16),
        j2v.Number("value"),
        name="measurements",
        max_length=5,
    ),
    j2v.Category("diagnosis", target=True, max_vocab_size=2),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    embed=True,
    root="measurements",
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

Inspect the model before serving. The plot should show `diagnosis` as a target, which is the field the endpoint will predict from the request.

In [4]:
model.plot(detail=True)

Schema
measurements [array] embed
measurements
d_model=16  attention=mha  max_length=1  n_outputs=1  n_layers=1  n_heads=4  batch_size=8  parameters=24,935  arrays=2  
fields=3  targets=1  embeds=1  embed=True  n_linear=1
├── measurements [array] 6,608 params
│   measurements/measurements
│   attention=mha  max_length=5  n_outputs=1  n_layers=1  n_heads=4  n_linear=1
│   ├── name [category] 4,119 params
│   │   measurements/measurements/name
│   │   query=[*].measurements[*].name  pooling=query  max_vocab_size=16  topk=[]  weight=1.0  n_heads=4  n_linear=1  
│   │   n_bands=8  p_unavailable=0.01
│   └── value [number] 4,007 params
│       measurements/measurements/value
│       query=[*].measurements[*].value  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  
│       n_bands=8  offset=4
└── diagnosis [category] target 3,593 params
    measurements/diagnosis
    query=[*].diagnosis  pooling=query  max_vocab_size=2  topk=[]  weight=1.0  n_heads=4  p_prune=1.0  n_linear=1  
    n_bands=8  p_unavailable=0.01

For serving, keep `diagnosis` as `target=True`. In prediction mode, target fields are supplied to the model as empty masked fields and decoded as outputs, so incoming requests do not need to include the answer you want the model to predict.

In [5]:
deployment = Deployment(model=model).forge(request=Request)

The final cell is guarded by an environment variable so documentation builds configure the server object without starting a long-running process.

In [6]:
if os.environ.get("JSON2VEC_SERVE") == "1":
    deployment.serve()